# Teste Gemini API — geração de exemplos N1

Este notebook é autónomo: não precisa do Excel nem das prompts já geradas.

Antes de executar, abre **Secrets** no Colab, cria `GEMINI_API_KEY` e ativa o acesso do notebook.

In [1]:
# ============================================================
# CONFIGURAÇÃO
# ============================================================

MODEL_NAME = "gemini-3.1-flash-lite"

NUMBER_OF_EXAMPLES = 2

OUTPUT_DIR = "/content/gemini_outputs"
OUTPUT_FILE = (
    f"{OUTPUT_DIR}/n1_test_{NUMBER_OF_EXAMPLES}_examples.json"
)

DOMAIN = "Bibliotecas"
FACT_TYPE = "Capacidade"
ENTITY_TYPE = "Biblioteca"
CONTEXT_LENGTH = "curto, com 3 a 5 frases"
FACT_POSITION = "no fim do contexto"
QUESTION_TYPE = "pergunta direta"
DISTRACTOR_RULE = (
    "não incluir informação enganadora nem distratores relevantes"
)

print("Modelo:", MODEL_NAME)
print("Exemplos:", NUMBER_OF_EXAMPLES)
print("Output:", OUTPUT_FILE)

Modelo: gemini-3.1-flash-lite
Exemplos: 2
Output: /content/gemini_outputs/n1_test_2_examples.json


In [2]:
!pip -q install -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 984.2/984.2 kB 24.8 MB/s eta 0:00:00


In [3]:
import json, re
from pathlib import Path
from google import genai
from google.genai import types
from google.colab import userdata
api_key = userdata.get("GEMINI_API_KEY")
if not api_key:
    raise ValueError("Secret GEMINI_API_KEY não encontrado.")
client = genai.Client(api_key=api_key)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print("Cliente Gemini criado.")

Cliente Gemini criado.


In [4]:
from google import genai
from google.colab import userdata

client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY")
)

for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-omni-flash-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
mod

In [5]:
response = client.models.generate_content(
    model=MODEL_NAME,
    contents="Responde apenas com a palavra: Olá"
)
print(response.text)

Olá


## Prompt fixa de teste

A prompt usa linguagem simples e não envia códigos internos da framework.

In [6]:
prompt = f"""
Gera exatamente {NUMBER_OF_EXAMPLES} exemplos em português europeu.

OBJETIVO
Criar exemplos para treinar um modelo pequeno a localizar um facto explícito num contexto fornecido.

DISTRIBUIÇÃO
Todos os exemplos devem ter estas características:
- domínio: {DOMAIN}
- tipo de facto pedido: {FACT_TYPE}
- tipo de entidade principal: {ENTITY_TYPE}
- contexto: {CONTEXT_LENGTH}
- posição do facto que responde à pergunta: {FACT_POSITION}
- tipo de pergunta: {QUESTION_TYPE}
- distratores: {DISTRACTOR_RULE}

REGRAS
1. Cada exemplo deve conter exatamente os campos context, question e answer.
2. A resposta deve estar claramente escrita no contexto.
3. Deve existir apenas uma resposta correta.
4. A pergunta deve pedir apenas um facto.
5. Não uses conhecimento externo.
6. Usa entidades realistas, preferencialmente inventadas.
7. Não repitas exemplos.
8. Devolve apenas um array JSON válido.
9. Não uses blocos Markdown.
10. Não acrescentes explicações antes ou depois do JSON.

FORMATO
[
  {{
    "context": "...",
    "question": "...",
    "answer": "..."
  }}
]
""".strip()
print(prompt)

Gera exatamente 2 exemplos em português europeu.

OBJETIVO
Criar exemplos para treinar um modelo pequeno a localizar um facto explícito num contexto fornecido.

DISTRIBUIÇÃO
Todos os exemplos devem ter estas características:
- domínio: Bibliotecas
- tipo de facto pedido: Capacidade
- tipo de entidade principal: Biblioteca
- contexto: curto, com 3 a 5 frases
- posição do facto que responde à pergunta: no fim do contexto
- tipo de pergunta: pergunta direta
- distratores: não incluir informação enganadora nem distratores relevantes

REGRAS
1. Cada exemplo deve conter exatamente os campos context, question e answer.
2. A resposta deve estar claramente escrita no contexto.
3. Deve existir apenas uma resposta correta.
4. A pergunta deve pedir apenas um facto.
5. Não uses conhecimento externo.
6. Usa entidades realistas, preferencialmente inventadas.
7. Não repitas exemplos.
8. Devolve apenas um array JSON válido.
9. Não uses blocos Markdown.
10. Não acrescentes explicações antes ou depois do

In [7]:
response = client.models.generate_content(
    model=MODEL_NAME,
    contents=prompt,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        temperature=0.8,
    ),
)
raw_text = response.text.strip()
print(raw_text[:1500])

[
  {
    "context": "A Biblioteca Municipal de Vila Nova de Gaia renovou recentemente as suas instalações. O espaço dispõe agora de zonas de estudo silenciosas e acesso gratuito a computadores. A biblioteca tem capacidade para acolher cento e cinquenta utilizadores em simultâneo.",
    "question": "Quantos utilizadores pode a Biblioteca Municipal de Vila Nova de Gaia acolher em simultâneo?",
    "answer": "Cento e cinquenta utilizadores"
  },
  {
    "context": "A Biblioteca Central de Évora abriu as portas ao público após um período de obras. O edifício histórico oferece diversas salas de leitura e uma cafetaria no piso térreo. Esta infraestrutura tem capacidade para albergar sessenta mil volumes na sua coleção permanente.",
    "question": "Qual é a capacidade de armazenamento de volumes da Biblioteca Central de Évora?",
    "answer": "Sessenta mil volumes"
  }
]


In [8]:
def clean_json_text(text):
    text = text.strip()
    if text.startswith("```json"):
        text = text[len("```json"):]
    if text.startswith("```"):
        text = text[len("```"):]
    if text.endswith("```"):
        text = text[:-3]
    return text.strip()

clean_text = clean_json_text(raw_text)
try:
    examples = json.loads(clean_text)
except json.JSONDecodeError as exc:
    print(clean_text)
    raise ValueError(f"JSON inválido: {exc}") from exc
if not isinstance(examples, list):
    raise TypeError("O resultado deve ser um array JSON.")
if len(examples) != NUMBER_OF_EXAMPLES:
    raise ValueError(f"Esperava {NUMBER_OF_EXAMPLES}, recebi {len(examples)}.")
required = {"context","question","answer"}
for i, ex in enumerate(examples,1):
    missing = required - set(ex)
    if missing:
        raise ValueError(f"Exemplo {i} sem campos: {sorted(missing)}")
print(f"JSON válido: {len(examples)} exemplos.")

JSON válido: 2 exemplos.


In [9]:
def normalize(text):
    text = re.sub(r"\s+", " ", text.lower().strip())
    return text.strip(" .,:;!?")
for i, ex in enumerate(examples,1):
    ok = normalize(ex["answer"]) in normalize(ex["context"])
    print(f"Exemplo {i}: resposta no contexto = {ok}")

Exemplo 1: resposta no contexto = True
Exemplo 2: resposta no contexto = True


In [10]:
for i, ex in enumerate(examples,1):
    print("\n" + "="*80)
    print(f"EXEMPLO {i}")
    print("CONTEXTO:", ex["context"])
    print("PERGUNTA:", ex["question"])
    print("RESPOSTA:", ex["answer"])


EXEMPLO 1
CONTEXTO: A Biblioteca Municipal de Vila Nova de Gaia renovou recentemente as suas instalações. O espaço dispõe agora de zonas de estudo silenciosas e acesso gratuito a computadores. A biblioteca tem capacidade para acolher cento e cinquenta utilizadores em simultâneo.
PERGUNTA: Quantos utilizadores pode a Biblioteca Municipal de Vila Nova de Gaia acolher em simultâneo?
RESPOSTA: Cento e cinquenta utilizadores

EXEMPLO 2
CONTEXTO: A Biblioteca Central de Évora abriu as portas ao público após um período de obras. O edifício histórico oferece diversas salas de leitura e uma cafetaria no piso térreo. Esta infraestrutura tem capacidade para albergar sessenta mil volumes na sua coleção permanente.
PERGUNTA: Qual é a capacidade de armazenamento de volumes da Biblioteca Central de Évora?
RESPOSTA: Sessenta mil volumes


In [11]:
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(examples, f, ensure_ascii=False, indent=2)
print("Guardado em:", OUTPUT_FILE)

Guardado em: /content/gemini_outputs/n1_test_2_examples.json


In [12]:
from google.colab import files
files.download(OUTPUT_FILE)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Próximo teste

Depois de confirmar que os 2 exemplos estão corretos, altera `NUMBER_OF_EXAMPLES` para `20` e executa novamente.